In [28]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [29]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def clean_json(text):
    """Strip markdown fences if Claude adds them anyway."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        text = text.rsplit("```", 1)[0]
    return text.strip()


def chat(messages, system_prompt=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system_prompt:
        params["system"] = system_prompt
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return "".join(block.text for block in message.content if block.type == "text")

    

In [30]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    text = chat(
        messages,
        system_prompt="Respond with a raw JSON array only. No markdown fences, no explanation."
    )
    return json.loads(clean_json(text))

In [ ]:
dataset = generate_dataset()
print(dataset)

# Once we have our dataset, we'll save it to a file so we can easily load it later during evaluation.
# This creates a dataset.json file in the same directory as your notebook, containing your list of tasks ready for prompt evaluation.
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

[{'task': 'Write a regex pattern that matches valid AWS S3 bucket names (3-63 characters, lowercase letters, numbers, hyphens, must start and end with lowercase letter or number)'}, {'task': 'Write a Python function that takes an AWS CloudFormation template as a JSON string and returns a list of all resource logical IDs'}, {'task': "Write a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'"}]
